# 103 — Reach Visualiser for `102-Simulate10Next.py`

Animates the full reach computed by `_02_pre_all` + `_02_get_all_opportunities`.

| Color | `ships_sent` |
|-------|-------------|
| Light blue `#4FC3F7` | 4 |
| Green `#81C784` | 16 |
| Orange `#FFB74D` | 64 |
| Pink `#F06292` | 256 |

Each animation frame = one arrival step. Lines go from source planet → target planet position at that step.

In [1]:
import importlib.util, copy, json, types, sys, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from IPython.display import HTML

spec = importlib.util.spec_from_file_location("agent102", "102-Simulate10Next.py")
m    = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
m.step = 0; m.num_agents = None; m.player_id = None

StrategyPipeline = m.StrategyPipeline
interpreter      = m._interpreter
print(f"Loaded agent102, NB_STEPS_SIM={m.GameConfig.NB_STEPS_SIM}")

Loaded agent102, NB_STEPS_SIM=10


In [2]:
LOG_NAME = "77588591"

replay       = json.load(open(f"62-logs/{LOG_NAME}.json", encoding="utf-8"))
steps_replay = replay["steps"]
print(f"Replay '{LOG_NAME}': {len(steps_replay)} steps, rewards={replay['rewards']}")

def obs_from_step(step_data, player=0):
    obs_dict = copy.deepcopy(step_data[player]["observation"])
    return types.SimpleNamespace(**obs_dict)

# ── Pick which replay step to analyse ────────────────────────────────────────
STEP = 0
obs  = obs_from_step(steps_replay[STEP], player=0)

owners     = {p[1] for p in obs.initial_planets if p[1] != -1}
num_agents = 4 if len(owners) > 2 else 2
env_step   = obs.step

print(f"Using replay step {STEP}  (env_step={env_step})")
print(f"Planets : {len(obs.planets)}")
print(f"Agents  : {num_agents}")
print(f"ω       : {obs.angular_velocity:.5f} rad/step")

Replay '77588591': 247 steps, rewards=[-1, 1]
Using replay step 0  (env_step=0)
Planets : 28
Agents  : 2
ω       : 0.02593 rad/step


In [3]:
# ── Compute reach from ALL planets for ships_sent ∈ [4, 16, 64, 256] ────────
SHIPS_LIST = [4, 16, 64, 256]

df_s, planet_disp = StrategyPipeline._01_get_obs_dataframe(obs, step=env_step, num_agents=num_agents)
coarse = StrategyPipeline._02_pre_all(df_s, ships_list=SHIPS_LIST)
pa     = StrategyPipeline._02_get_all_opportunities(coarse, df_s, planet_disp)

print(f"Total reach rows : {len(pa)}")
print(f"Arrival steps    : {sorted(pa['step'].unique())}")
print(f"Ships_sent levels: {sorted(pa['ships_sent'].unique())}")

Total reach rows : 1296
Arrival steps    : [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Ships_sent levels: [np.int64(4), np.int64(16), np.int64(64), np.int64(256)]


In [4]:
# ── All reach at step 0 (all opportunities sourced from observation at step=0) ─
PRINT_COLS = ["id_src", "x_src", "y_src", "ships_sent", "step", "id", "x", "y", "angle"]
print(f"Full reach table from step 0 — {len(pa)} rows\n")
with pd.option_context('display.max_rows', None, 'display.float_format', '{:.2f}'.format):
    display(pa[PRINT_COLS].sort_values(["step", "id_src", "ships_sent", "id"]).reset_index(drop=True))

Full reach table from step 0 — 1296 rows



,id_src,x_src,y_src,ships_sent,step,id,x,y,angle
0,12,69.41,79.21,64,2,24,78.41,76.55,-0.29
1,12,69.41,79.21,256,2,20,65.04,70.66,-2.02
2,12,69.41,79.21,256,2,24,78.41,76.55,-0.30
3,13,20.79,69.41,64,2,25,23.45,78.41,1.28
4,13,20.79,69.41,256,2,21,29.34,65.04,-0.45
5,13,20.79,69.41,256,2,25,23.45,78.41,1.27
6,14,79.21,30.59,64,2,26,76.55,21.59,-1.86
7,14,79.21,30.59,256,2,22,70.66,34.96,2.69
8,14,79.21,30.59,256,2,26,76.55,21.59,-1.87
9,15,30.59,20.79,64,2,27,21.59,23.45,2.85


In [5]:
# ── Reach animation ──────────────────────────────────────────────────────────
SHIPS_COLORS = {4: "#4FC3F7", 16: "#81C784", 64: "#FFB74D", 256: "#F06292"}
OWNER_COLORS = {0: "steelblue", 1: "tomato", -1: "#888888"}


def make_reach_animation(pa, df_s, ships_list, title="Reach map", interval=500):
    planet_info = (
        df_s.set_index(["id", "step"])[["x", "y", "radius", "ships", "production", "owner"]]
    )
    steps = sorted(pa["step"].unique())

    # viewport: tight around all planet positions + margin
    margin = 5
    xs = df_s["x"].values
    ys = df_s["y"].values
    x_min, x_max = xs.min() - margin, xs.max() + margin
    y_min, y_max = ys.min() - margin, ys.max() + margin

    legend_patches = [
        mpatches.Patch(color=SHIPS_COLORS[s], label=f"{s} ships") for s in ships_list
    ]

    fig, ax = plt.subplots(figsize=(8, 8))
    fig.patch.set_facecolor("#111122")

    def draw(i):
        step = steps[i]
        ax.cla()
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_max, y_min)   # flipped to match game coords
        ax.set_aspect("equal")
        ax.set_facecolor("#111122")
        ax.tick_params(colors="#aaaaaa")
        for sp in ax.spines.values():
            sp.set_edgecolor("#444444")
        ax.set_title(f"{title} — arrival step {step}", color="white", fontsize=12)

        if x_min < 50 < x_max and y_min < 50 < y_max:
            ax.add_patch(plt.Circle((50, 50), 10, color="gold", zorder=2, alpha=0.9))

        # Reach lines per ships_sent level
        pa_step = pa[pa["step"] == step]
        for s, color in SHIPS_COLORS.items():
            sub = pa_step[pa_step["ships_sent"] == s]
            if sub.empty:
                continue
            segs = [[(r.x_src, r.y_src), (r.x, r.y)] for r in sub.itertuples()]
            ax.add_collection(LineCollection(segs, colors=color, alpha=0.55, linewidth=1.8, zorder=4))
            ax.scatter(sub["x"].values, sub["y"].values, color=color, s=18, alpha=0.7, zorder=5)

        # Planets on top
        for pid in df_s["id"].unique():
            try:
                info = planet_info.loc[(pid, step)]
            except KeyError:
                continue
            x, y, radius = info["x"], info["y"], info["radius"]
            ships, production, owner = int(info["ships"]), int(info["production"]), int(info["owner"])
            c = OWNER_COLORS.get(owner, "#888888")
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.9, zorder=6))
            ax.text(x, y, str(ships), ha="center", va="center",
                    color="white", fontsize=8, fontweight="bold", zorder=7)
            ax.text(x, y - radius - 0.4, f"#{int(pid)}",
                    ha="center", va="top", color="#cccccc", fontsize=7, zorder=7)

        ax.legend(handles=legend_patches, loc="lower right",
                  facecolor="#222233", edgecolor="#555555", labelcolor="white", fontsize=9)
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(steps), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())


make_reach_animation(pa, df_s, SHIPS_LIST, title=f"{LOG_NAME} step {STEP}", interval=500)